# BitRoss 2 · 1/3 · Curate the dataset

Downloads `OVAWARE/16xModdedMinecraft` (~1.03M textures), measures every image, filters junk
(PBR/overlay maps, fragments, flat fills, exact duplicates, recolor spam, per-mod caps), builds the
1024-color OKLab palette and writes everything to Google Drive.

**Runtime:** any (CPU is fine, ~10 min with 8 cores). Outputs go to `DRIVE_ROOT/work`.

In [ ]:
#@title Setup: Drive, code, Hugging Face login
#@markdown Put your Hugging Face token in Colab **Secrets** (key icon in the left bar) as `HF_TOKEN`
#@markdown and enable notebook access. The dataset is private, so the token is required.
REPO_URL = "https://github.com/OVAWARE/BitRoss"  #@param {type:"string"}
BRANCH = "v2"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/BitRoss2"  #@param {type:"string"}

import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
if not os.path.exists("/content/BitRoss"):
    code_zip = f"{DRIVE_ROOT}/code.zip"
    if os.path.exists(code_zip):            # fallback: upload a zip of the repo to Drive
        subprocess.run(["unzip", "-q", code_zip, "-d", "/content/BitRoss"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, "/content/BitRoss"], check=True)
os.chdir("/content/BitRoss")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!pip -q install -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title Download the dataset
from huggingface_hub import snapshot_download
raw = snapshot_download("OVAWARE/16xModdedMinecraft", repo_type="dataset", local_dir="/content/raw",
                        token=os.environ["HF_TOKEN"])
!ls -la /content/raw/data

In [ ]:
#@title Scan → filter → palette → previews  (work dir on local disk, then copied to Drive)
!python -m bitross.data.build scan --parquet "/content/raw/data/*.parquet" --work /content/work
!python -m bitross.data.build filter --work /content/work
!python -m bitross.data.build palette --work /content/work --palette-size 1024 --translucent 64 --device cuda
!python -m bitross.data.build preview --work /content/work
!mkdir -p {DRIVE_ROOT}/work && cp -r /content/work/* {DRIVE_ROOT}/work/
!cat /content/work/filter_summary.json

In [ ]:
#@title Look at what was kept and dropped
from IPython.display import Image, display
import glob
for p in sorted(glob.glob("/content/work/preview/*.png")):
    print(os.path.basename(p)); display(Image(p))

In [ ]:
#@title (Optional) build a training cache right away with filename captions only
#@markdown Skip this if you are going to run the VLM labeling notebook (recommended).
!python -m bitross.data.build cache --work /content/work --out /content/cache
!mkdir -p {DRIVE_ROOT}/cache && cp -r /content/cache/* {DRIVE_ROOT}/cache/